# 02 — Every pairwise comparison of cell counts, one cell type at a time

`01` asked whether the *whole composition vector* shifts together. That is a summary statistic,
and a summary statistic can be flat while one cell type moves a lot. This notebook does the
opposite: **every group against every other group, every cell type separately, with a plain
unadjusted p-value next to the actual percentages.** Nothing is collapsed.

10 groups (5 arms x 2 tumour sites) = 45 pairs x 18 cell types = **810 comparisons**, all
printed, all saved. The point is to look at everything before concluding anything.

### What gets reported for each comparison
- mean **cell count** and mean **percentage** in each group (both, because they answer different
  questions — see the note in §2)
- `log2FC_prop` — fold change of the percentage
- `p_prop` — Welch t-test on log percentage, **unadjusted**; `p_clr` — the same test after the
  closure correction; `p_t` is the smaller of the two, used only for sorting. **`scales_agree`
  flags where they disagree** — that is a signal the change is driven by something else in the
  sample, not a nuisance to be averaged away. `p_mw` is Mann-Whitney (its floor is 0.1 at 3v3)
- `separated` — every sample in one group above every sample in the other. At n = 3 vs 3 this
  happens 1 time in 20 by chance, so it carries about as much weight as p = 0.05, and it does
  not assume normality
- `min_cells` — the smallest count in any sample of either group. **Below ~20 the comparison is
  counting noise**, whatever the p-value says

### On `NoTx`
It is in the matrix for completeness and it is *not* a comparator for any RT claim. Untreated
tumours differ from treated ones in nearly everything (tumour is 25-42% of cells vs 4-6%), so a
difference against `NoTx` tells you the animal got treated, not what radiation did. Every claim
below rests on **treated-vs-treated** comparisons.

### The three questions the matrix actually answers
| question | comparison |
|---|---|
| what does RT add at the irradiated site? | `RTCyPSCA_Tu1` vs `CyPSCA_Tu1`, `RTCyTAG72_Tu1` vs `CyTAG72_Tu1` |
| what does RT add at the abscopal site? | `RTCyPSCA_Tu2` vs `CyPSCA_Tu2`, `RTCyTAG72_Tu2` vs `CyTAG72_Tu2` |
| does the CAR target matter more than the beam? | `RTCyPSCA_*` vs `RTCyTAG72_*` |

## 1. Config and load

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import itertools
import numpy as np, pandas as pd, h5py, anndata as ad
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 300); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)

_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T"); _LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _HPC if _HPC.exists() else _LOC
SC_REF = (PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad"
          if PROJ is _HPC else PROJ/"data/sc-reference/annotation_v4.h5ad")
OUT = PROJ/"results/rt_advantage"; FIG = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

MIN_CELLS_TRUST = 20     # below this the comparison is counting noise
PSEUDO = 0.5

with h5py.File(SC_REF, "r") as f:
    o = ad.io.read_elem(f["obs"])
o["sample"] = o["file_name"].astype(str)
p = o["sample"].str.extract(r"^(?P<arm>.+)_Tu(?P<site>\d)_(?P<rep>\d+)$")
o["arm"], o["site"] = p["arm"], p["site"].astype(int)
o["mouse"] = p["arm"] + "_m" + p["rep"]
o["ct"] = o["cell_type_lvl2"].astype(str)

cnt  = pd.crosstab(o["sample"], o["ct"]).drop(columns=["Erythrocyte"], errors="ignore")
meta = o.groupby("sample", observed=True).agg(arm=("arm","first"), site=("site","first"),
                                              mouse=("mouse","first"))
meta["grp"] = meta.arm + "_Tu" + meta.site.astype(str)
cnt  = cnt.reindex(meta.index)
prop = cnt.div(cnt.sum(1), axis=0)
x = cnt.values.astype(float) + PSEUDO
lg = np.log(x / x.sum(1, keepdims=True))
CLR = pd.DataFrame(lg - lg.mean(1, keepdims=True), index=cnt.index, columns=cnt.columns)

print(f"{len(cnt)} samples x {cnt.shape[1]} cell types")
print(meta.groupby("grp").size().to_string())
cnt.join(meta).to_csv(OUT/"cell_counts_per_sample.csv")

## 2. The raw counts, printed

Both counts and percentages are shown throughout, and they mean different things:

* **Percentages are the interpretable quantity in dissociated scRNA-seq.** How many cells of a
  type you recover depends on how many cells were loaded and how well that type survived
  dissociation, so an absolute count is not a density.
* **Counts still matter** — as the reliability check. A 3-fold change in something present at
  1 cell per sample is noise; the same change at 500 cells is not.

In [ ]:
print("ABSOLUTE CELL COUNTS per sample (Erythrocyte removed from the total)")
print(cnt.join(meta[["grp"]]).set_index("grp", append=True).to_string())
print("\n\nPERCENT of cells per sample")
print((prop*100).round(2).join(meta[["grp"]]).set_index("grp", append=True).to_string())

## 3. All 45 x 18 comparisons

In [ ]:
GRPS = sorted(meta.grp.unique())
rows = []
for a, b in itertools.combinations(GRPS, 2):
    ia, ib = meta.index[meta.grp == a], meta.index[meta.grp == b]
    for ct in cnt.columns:
        va, vb = CLR.loc[ia, ct].values, CLR.loc[ib, ct].values
        pa, pb = prop.loc[ia, ct].values, prop.loc[ib, ct].values
        ca, cb = cnt.loc[ia, ct].values, cnt.loc[ib, ct].values
        # two scales, reported side by side on purpose:
        #   p_prop  - on log(percentage). Matches log2FC_prop and `separated`, but percentages
        #             are closed, so a big influx elsewhere can move it on its own.
        #   p_clr   - closure-corrected. Does NOT have to agree with `separated`, because CLR
        #             divides by the sample's geometric mean and can reorder samples.
        # When they disagree, look at what else moved in that sample.
        _t, p_clr  = stats.ttest_ind(va, vb, equal_var=False)
        _t, p_prop = stats.ttest_ind(np.log(pa + 1e-6), np.log(pb + 1e-6), equal_var=False)
        try: _u, pu = stats.mannwhitneyu(pa, pb, alternative="two-sided")
        except Exception: pu = np.nan
        rows.append(dict(A=a, B=b, cell_type=ct, nA=len(ia), nB=len(ib),
                         cntA=ca.mean(), cntB=cb.mean(),
                         pctA=100*pa.mean(), pctB=100*pb.mean(),
                         log2FC_prop=np.log2((pa.mean()+1e-9)/(pb.mean()+1e-9)),
                         clr_delta=va.mean()-vb.mean(),
                         p_prop=p_prop, p_clr=p_clr, p_t=min(p_prop, p_clr), p_mw=pu,
                         scales_agree=bool((p_prop < 0.1) == (p_clr < 0.1)),
                         separated=bool(pa.min() > pb.max() or pa.max() < pb.min()),
                         min_cells=int(min(ca.min(), cb.min()))))
R = pd.DataFrame(rows)
R.to_csv(OUT/"pairwise_celltype_comparisons.csv", index=False)

n_exp = len(R)*0.05
print(f"{len(R)} comparisons. At p<0.05 by chance you would expect ~{n_exp:.0f}.")
print(f"observed p<0.05: {(R.p_t<0.05).sum()}   p<0.01: {(R.p_t<0.01).sum()}   "
      f"separated: {R.separated.sum()}")
trust = R[R.min_cells >= MIN_CELLS_TRUST]
print(f"\nrestricted to comparisons where every sample had >= {MIN_CELLS_TRUST} cells: "
      f"{len(trust)} tests, p<0.05 in {(trust.p_t<0.05).sum()} (chance ~{len(trust)*0.05:.0f})")
print("\nThat excess is real but it is almost entirely the treatment effect — see §4.")

## 4. Where the signal actually sits: treated vs treated only

Dropping every comparison that involves `NoTx` is the single most informative filter in this
notebook. What is left is the set of differences that could be attributable to radiation or to
the CAR target, rather than to having been treated at all.

In [ ]:
noTx = trust[trust.A.str.startswith("NoTx") | trust.B.str.startswith("NoTx")]
tvt  = trust[~(trust.A.str.startswith("NoTx") | trust.B.str.startswith("NoTx"))]
print(f"comparisons involving NoTx : {len(noTx):>4}, p<0.05 in {(noTx.p_t<0.05).sum():>3} "
      f"({100*(noTx.p_t<0.05).mean():.0f}%)")
print(f"treated vs treated only    : {len(tvt):>4}, p<0.05 in {(tvt.p_t<0.05).sum():>3} "
      f"({100*(tvt.p_t<0.05).mean():.0f}%, chance 5%)")

print("\nTREATED-VS-TREATED HITS, p<0.10, >=20 cells in every sample")
print(tvt[tvt.p_t < 0.10].sort_values("p_t")[
    ["A","B","cell_type","pctA","pctB","log2FC_prop","p_prop","p_clr","scales_agree","separated","min_cells"]
    ].round(3).to_string(index=False))

## 5. The six comparisons that were asked for, in full

Each printed with every cell type, sorted by p, so nothing is hidden behind a threshold.

In [ ]:
FOCUS = [("RTCyPSCA_Tu1",  "CyPSCA_Tu1",     "RT effect, irradiated site, PSCA CAR"),
         ("RTCyTAG72_Tu1", "CyTAG72_Tu1",    "RT effect, irradiated site, TAG72 CAR"),
         ("RTCyPSCA_Tu2",  "CyPSCA_Tu2",     "RT effect, ABSCOPAL site, PSCA CAR"),
         ("RTCyTAG72_Tu2", "CyTAG72_Tu2",    "RT effect, ABSCOPAL site, TAG72 CAR"),
         ("RTCyPSCA_Tu1",  "RTCyPSCA_Tu2",   "irradiated vs abscopal within the RT PSCA arm"),
         ("RTCyTAG72_Tu1", "RTCyTAG72_Tu2",  "irradiated vs abscopal within the RT TAG72 arm"),
         ("RTCyTAG72_Tu1", "RTCyPSCA_Tu1",   "CAR target under RT, irradiated"),
         ("RTCyPSCA_Tu2",  "RTCyTAG72_Tu2",  "CAR target under RT, abscopal")]


def show(a, b, note=""):
    sub = R[((R.A == a) & (R.B == b)) | ((R.A == b) & (R.B == a))].copy()
    if sub.empty:
        print(f"  {a} vs {b}: not in the matrix"); return
    if sub.A.iloc[0] != a:                      # orient so positive = higher in `a`
        sub["log2FC_prop"] *= -1; sub["clr_delta"] *= -1
        sub = sub.rename(columns={"pctA":"pctB","pctB":"pctA","cntA":"cntB","cntB":"cntA"})
    sub["reliable"] = np.where(sub.min_cells >= MIN_CELLS_TRUST, "", "  <-too few cells")
    print(f"\n{'='*118}\n{a}  vs  {b}   —   {note}\n"
          f"positive log2FC = higher in {a}\n{'='*118}")
    print(sub.sort_values("p_t")[["cell_type","cntA","cntB","pctA","pctB","log2FC_prop",
                                  "p_prop","p_clr","scales_agree","separated","min_cells","reliable"]]
             .round(3).to_string(index=False))


for a, b, note in FOCUS:
    show(a, b, note)

## 6. The control that kills most of it: is `Tu1 > Tu2` an RT effect or a site effect?

Any `Tu1` vs `Tu2` difference inside an RT arm is only about the beam if the **non-RT arms do
not show the same difference**. This prints the irradiated/abscopal ratio for every arm side by
side. Read down the column: if `CyPSCA` and `CyTAG72` show the same gap as `RTCyPSCA` and
`RTCyTAG72`, the beam is not what made it.

In [ ]:
ARM_ORDER = ["NoTx", "CyPSCA", "CyTAG72", "RTCyPSCA", "RTCyTAG72"]
pc = (prop*100).join(meta)
site_rows = []
for ct in cnt.columns:
    for arm in ARM_ORDER:
        a = pc[(pc.arm == arm) & (pc.site == 1)][ct]
        b = pc[(pc.arm == arm) & (pc.site == 2)][ct]
        if len(a) < 2 or len(b) < 2: continue
        _t, pp = stats.ttest_ind(np.log(a+.01), np.log(b+.01), equal_var=False)
        site_rows.append(dict(cell_type=ct, arm=arm, Tu1=a.mean(), Tu2=b.mean(),
                              ratio=a.mean()/max(b.mean(), 1e-9), p=pp,
                              separated=bool(a.min() > b.max() or a.max() < b.min()),
                              min_cells=int(min(cnt.loc[a.index, ct].min(),
                                                cnt.loc[b.index, ct].min()))))
S = pd.DataFrame(site_rows)
S.to_csv(OUT/"tu1_vs_tu2_by_arm.csv", index=False)

for ct in ["CD8_T","CD4_T","NK","Treg","M1_like_Mac","M2_like_Mac","Intermediate_Mac",
           "Classical_Mono","Cancer_cell","N1_like_Neu"]:
    d = S[S.cell_type == ct].set_index("arm").reindex(ARM_ORDER).dropna(how="all")
    print(f"\n--- {ct}: irradiated (Tu1) vs abscopal (Tu2), % of cells ---")
    print(d[["Tu1","Tu2","ratio","p","separated","min_cells"]].round(3).to_string())
    rt = d.loc[[a for a in ["RTCyPSCA","RTCyTAG72"] if a in d.index], "ratio"]
    nort = d.loc[[a for a in ["CyPSCA","CyTAG72"] if a in d.index], "ratio"]
    if len(rt) and len(nort):
        verdict = ("RT-SPECIFIC — the gap is absent without the beam"
                   if nort.mean() < 1.15 and rt.mean() > 1.3 else
                   "NOT RT-specific — the non-RT arms show the same gap")
        print(f"    mean ratio  RT arms {rt.mean():.2f}  |  non-RT CAR-T arms {nort.mean():.2f}"
              f"   -> {verdict}")

## 7. Per-mouse view of the two recurring leads

Numbers averaged over three mice hide whether all three moved. These are the individual animals
for the two cell types that keep surfacing: `CD8_T` and `M1_like_Mac`. `Tu1` and `Tu2` come from
the same animal, so each row is a paired observation.

In [ ]:
LEADS = ["CD8_T", "M1_like_Mac", "M2_like_Mac"]
w = (pc.reset_index().pivot_table(index=["arm","mouse"], columns="site", values=LEADS))
w.columns = [f"{a}_Tu{b}" for a, b in w.columns]
for ct in LEADS:
    w[f"{ct}_ratio"] = w[f"{ct}_Tu1"] / w[f"{ct}_Tu2"]
print("percent of cells, per mouse (Tu1 = irradiated, Tu2 = abscopal)")
print(w.round(2).to_string())
w.to_csv(OUT/"per_mouse_leads.csv")

fig, axes = plt.subplots(1, len(LEADS), figsize=(5.2*len(LEADS), 4.6))
for ax, ct in zip(np.atleast_1d(axes), LEADS):
    for i, arm in enumerate(ARM_ORDER):
        d = w.loc[arm] if arm in w.index.get_level_values(0) else None
        if d is None: continue
        for _m, r in d.iterrows():
            ax.plot([i-0.16, i+0.16], [r[f"{ct}_Tu1"], r[f"{ct}_Tu2"]],
                    color="#d1495b" if arm.startswith("RT") else "#3b6ea5", marker="o", ms=5,
                    lw=1.4, alpha=.85)
    ax.set_xticks(range(len(ARM_ORDER))); ax.set_xticklabels(ARM_ORDER, rotation=35, ha="right")
    ax.set_ylabel("% of cells"); ax.set_title(f"{ct}   (left dot = Tu1 irradiated,\nright = Tu2 abscopal)",
                                              fontsize=10)
    ax.grid(axis="y", alpha=.3)
fig.suptitle("One line per mouse. Red = RT arms. A beam effect needs the red lines to tilt "
             "where the blue ones do not.", fontsize=11)
fig.tight_layout(); fig.savefig(FIG/"02_per_mouse_leads.png", dpi=250, bbox_inches="tight")
plt.show()

## 8. Which leads are worth a differential-expression follow-up?

A population that shifts in *abundance* is worth asking whether it also shifted in *state* — are
these the same cells, just more of them, or different cells? A candidate qualifies only if:

1. the comparison is **treated vs treated** (so it is not the treatment effect),
2. `p_t < 0.15` **or** fully separated,
3. **>= 100 cells in every sample**, which is what pseudobulk DE at n = 3 a side needs.

The output is a ranked shortlist, not a conclusion.

In [ ]:
cand = tvt[((tvt.p_t < 0.15) | tvt.separated) & (tvt.min_cells >= 100)].copy()
cand["abs_l2fc"] = cand.log2FC_prop.abs()
cand = cand.sort_values(["p_t", "abs_l2fc"])
print(f"{len(cand)} DE-follow-up candidates\n")
print(cand[["A","B","cell_type","pctA","pctB","log2FC_prop","p_prop","p_clr","separated","min_cells"]]
        .round(3).to_string(index=False))
cand.to_csv(OUT/"de_followup_candidates.csv", index=False)

print("\n\nGrouped by cell type — how often does each population surface?")
print(cand.groupby("cell_type").agg(n_comparisons=("p_t","size"), best_p=("p_t","min"),
                                    max_abs_log2FC=("abs_l2fc","max")).round(3).to_string())

is_rt = lambda a, b: (a.startswith("RT") != b.startswith("RT")) and a.split("_")[-1] == b.split("_")[-1]
cand["contrast_type"] = [
    "RT vs non-RT (same CAR, same site)" if is_rt(r.A, r.B)
    else "CAR target (PSCA vs TAG72)" if ("PSCA" in r.A) != ("PSCA" in r.B)
    else "site (Tu1 vs Tu2, same arm)" if r.A.split("_Tu")[0] == r.B.split("_Tu")[0]
    else "mixed" for _, r in cand.iterrows()]
print("\nwhat kind of contrast are the candidates?")
print(cand.contrast_type.value_counts().to_string())
print("\n-> if 'CAR target' dominates, the composition is tracking which CAR was given,")
print("   not whether the animal was irradiated.")

## 9. Summary

In [ ]:
print("="*100); print("SUMMARY — report these back"); print("="*100)
print(f"\n[scope] {len(R)} comparisons = 45 group pairs x {cnt.shape[1]} cell types")
print(f"  reliable (>={MIN_CELLS_TRUST} cells everywhere): {len(trust)}")
print(f"  comparisons where the two scales disagree: {int((~trust.scales_agree).sum())} "
      f"({100*(~trust.scales_agree).mean():.0f}%) — these are closure, look at what else moved")
print(f"  of those, treated-vs-treated: {len(tvt)}, p<0.05 in {(tvt.p_t<0.05).sum()} "
      f"(chance ~{len(tvt)*0.05:.0f})")

print("\n[RT vs matched non-RT, same CAR target, same site — the actual RT question]")
for a, b, note in FOCUS[:4]:
    sub = R[((R.A==a)&(R.B==b))|((R.A==b)&(R.B==a))]
    sub = sub[sub.min_cells >= MIN_CELLS_TRUST].nsmallest(3, "p_t")
    print(f"  {a} vs {b}:")
    for _, r in sub.iterrows():
        print(f"      {r.cell_type:<18s} {r.pctA:6.2f}% vs {r.pctB:6.2f}%  "
              f"log2FC {r.log2FC_prop:+.2f}  p_prop={r.p_prop:.3f} p_clr={r.p_clr:.3f}"
              f"{'  SEPARATED' if r.separated else ''}")

print("\n[site control] cell types where Tu1/Tu2 differs ONLY in the RT arms:")
rt_specific = []
for ct in cnt.columns:
    d = S[S.cell_type == ct].set_index("arm")
    if not {"RTCyPSCA","RTCyTAG72","CyPSCA","CyTAG72"} <= set(d.index): continue
    rt_r = d.loc[["RTCyPSCA","RTCyTAG72"], "ratio"].mean()
    no_r = d.loc[["CyPSCA","CyTAG72"], "ratio"].mean()
    mc = d.loc[["RTCyPSCA","RTCyTAG72","CyPSCA","CyTAG72"], "min_cells"].min()
    if rt_r > 1.3 and no_r < 1.15 and mc >= MIN_CELLS_TRUST:
        rt_specific.append((ct, rt_r, no_r))
print("  " + (", ".join(f"{c} (RT {r:.2f}x vs non-RT {n:.2f}x)" for c, r, n in rt_specific)
              or "NONE — every Tu1/Tu2 gap that is present with RT is also present without it"))

print(f"\n[DE follow-up shortlist] {len(cand)} candidates, by contrast type:")
for k, v in cand.contrast_type.value_counts().items(): print(f"    {k}: {v}")
print("\n  top 5:")
for _, r in cand.head(5).iterrows():
    print(f"    {r.cell_type:<18s} {r.A} vs {r.B}   {r.pctA:.2f}% vs {r.pctB:.2f}%  p={r.p_t:.3f}")

print(f"\n[files written to {OUT}]")
for f_ in sorted(OUT.glob("*.csv")): print(f"  {f_.name}")
print("="*100)